# Протокол проверки прогноза движения сердца по одной КТ

**Назначение.** Этот документ задаёт постановку проверки готовой модели движения и фиксирует, какая часть протокола выполнена на 4D-КТ добровольца Ника. Под «одной КТ» понимается один трёхмерный том фазы 00, а не отдельный двумерный срез. Результаты вычислительного опыта подробно представлены в [ноутбуке прогноза](21.19_MP07_Прогноз_по_одной_КТ.ipynb) и [пофазном отчёте](mp06_nix_one_ct_forecast_report.md).

**Научный вопрос.** Улучшает ли готовая модель, получив форму сердца из одной фазы и номинальную отметку сердечного цикла, совпадение с сердцем в других реально снятых фазах по сравнению с сохранением исходной формы неподвижной? Это проверка добавленной пользы модели на одном человеке. Полученные смещения остаются результатом расчёта; непосредственно измеренным движением соответствующих точек сердца они не являются.

**Статус на 01.10.2026.** Технический прогноз и сравнение с автоматическими масками восьми фаз выполнены. Анатомическая и прогностическая пригодность для FEM и оценки функций сердца не подтверждена.

## Данные и разделение фаз

Исследование выполнено на 4D-КТ Ника (DICOM-серия 307). Опорная фаза 00 содержит трёхмерный КТ-том на исходной сетке и автоматическую маску `whole_heart`, построенную TotalSegmentator v2. Из маски получена поверхность в миллиметрах: координаты NIfTI RAS переведены в используемую моделью систему LPS. **Индивидуальную форму** в опыте задаёт именно маска, полученная из КТ фазы 00; пиксели КТ после сегментации напрямую в подгонку формы и нейросеть движения не поступают. Готовые сердечные веса и модель формы взяты из исторического официального выпуска MONAI Physio `2026.07.3`. В расчёте используются 15 коэффициентов главных компонент формы; обучение или дообучение на данных проекта не проводилось.

Целевые фазы 01–08 имели номинальные отметки `23, 33, 43, 53, 62, 72, 82 и 92%` интервала R–R по метаданным КТ; исходная фаза 00 — `14%`. Эти числа подавались сети как доля цикла от 0 до 1. Они не заменяют независимую ЭКГ-разметку и не задают точно момент механического сокращения. Фазы с номинальной отметкой от 100% исключены из этого опыта, поскольку выходят за диапазон модели.

При подгонке индивидуальной формы использована только автоматическая маска, полученная из КТ фазы 00; сам том сохраняется как источник маски и для контроля происхождения данных. КТ фаз 01–08 использовали при их отдельной сегментации, а сохранённые поверхности этих фаз — для оценки уже рассчитанного прогноза. Таким образом, контрольные маски не входят в вычисление движущейся формы. Однако все маски получены одним автоматическим сегментатором; их анатомическая правильность вручную не принята.

## Построение прогноза и неподвижного контроля

Среднюю поверхность готовой модели сначала совместили с автоматической поверхностью Ника в фазе 00 аффинным преобразованием, затем по этой же фазе оценили 15 коэффициентов формы. Полученная **подогнанная поверхность фазы 00** служит общей исходной геометрией для двух вариантов сравнения. Она не тождественна исходной маске TotalSegmentator v2: модель имеет собственное анатомическое определение и ограниченный базис формы.

Для целевой доли цикла `t` готовая сеть рассчитала смещение вершин. Из него вычли расчётное смещение при исходной доле `0,14`; разность перенесли в координаты Ника и добавили к подогнанной поверхности. Тем самым сравнивалось предсказанное **изменение относительно фазы 00**, а не абсолютное положение исторического шаблона. Веса сети и подгонка формы оставались неизменными при проверке всех целевых фаз.

**Неподвижный контроль** — та же подогнанная поверхность фазы 00 без добавления сетевого смещения. Такой контроль заведомо не описывает реальный кардиоцикл. Его роль состоит в проверке, улучшает ли сетевое движение результат при одинаковой исходной анатомии и одинаковых данных, доступных до целевой фазы. Контрольной анатомией для обоих вариантов служили отдельно сегментированные поверхности реально снятых фаз 01–08.

Первоначально были выбраны фазы 01, 03, 07 и 08. Затем без изменения весов, подгонки и метрики анализ расширили на все восемь целевых фаз внутри диапазона сети. Это последовательное расширение отражено в [журнале опыта](mp06_nix_one_ct_forecast_report.md).

## Метрика и выполненные проверки

Основная метрика — **средняя симметричная дистанция ближайших вершин**, в миллиметрах. Для каждой вершины прогноза находили ближайшую вершину контрольной поверхности, затем повторяли поиск в обратном направлении; среднее двух направлений уменьшает зависимость результата от того, какую поверхность назначили первой. Меньшее значение означает более близкое расположение дискретизированных поверхностей. Итоговые значения пересчитаны по всем вершинам сохранённых сеток. Метрика зависит от плотности и распределения вершин; она **не устанавливает соответствие анатомических точек**, точность локальных векторов движения или правильность объёма камер.

Технический контроль включал проверку полного файла модели формы по SHA-256, порядка вершин шаблона, допустимости аффинного преобразования, фиксированного набора коэффициентов и неизменности сохранённых прогнозов при итоговом пересчёте. Среднее расхождение подогнанного шаблона с маской исходной фазы составило `4,384 мм` по всем вершинам. Оно ограничивает интерпретацию более мелких различий между вариантом с движением и неподвижным контролем.

Исходный план предусматривал отдельные ошибки положения центра, объёма, перекрытия масок, качества деформационного поля и влияния на импеданс. Прямой опыт с готовой сетью пока проверил **поверхностное совпадение**. Остальные критерии и одинаковая FEM-постановка для сетевого прогноза не выполнены; нельзя выдавать их за отрицательные или положительные результаты.

## Результат сопоставления с реальной 4D-КТ

Для восьми целевых фаз средняя дистанция до отдельно полученных автоматических контуров составила `4,509 мм` у неподвижной подогнанной формы и `4,483 мм` у сетевого прогноза. Разница `0,026 мм` составляет около `0,6%` от ошибки неподвижного контроля. На четырёх фазах прогноз приблизился к контрольной поверхности, на четырёх удалился. Например, в фазе 07 дистанция уменьшилась с `4,633` до `4,410 мм`, тогда как в фазе 01 выросла с `4,091` до `4,308 мм`. По всем фазам устойчивого практически значимого выигрыша не наблюдается. Показатели каждой фазы приведены в [подробной таблице](mp06_nix_one_ct_forecast_report.md).

**Интерпретация.** Сеть вычислила ненулевые смещения, но при выбранной проверке их использование почти не изменило среднее совпадение с 4D-КТ. Этот результат не означает отсутствия движения сердца: неподвижный вариант является нулевым прогнозом, а ошибка исходной подгонки составляет несколько миллиметров.

## Ограничения и решение о применении

Готовый исторический шаблон и наша маска TotalSegmentator v2 `whole_heart` могут описывать разные анатомические границы. Полученные контрольные контуры независимы от сетевого прогноза, но являются автоматическими и не заменяют принятую ручную разметку. Номинальный процент R–R не подтверждён как точная механическая временная координата. Также неизвестно, насколько вывод для одного добровольца переносится на других людей и протоколы съёмки. Числа поверхностной дистанции нельзя интерпретировать как погрешность индивидуальных траекторий вершин.

**Решение для текущей методики:** готовая модель запущена и проверена без обучения на собственных данных; убедимого улучшения по итоговой геометрической метрике нет. Её выход пока не передают в FEM как анатомически проверенную движущуюся геометрию и не используют для вывода об ударном объёме или фракции выброса. Для пересмотра нужны совместимое анатомическое определение формы, независимая контрольная разметка и проверка влияния ошибки геометрии на одинаковый электрический расчёт. Более широкую популяционную применимость нельзя заключить по одному добровольцу.

Воспроизводимый [скрипт](mp06_nix_one_ct_forecast.py) сохраняет индивидуальные поверхности, коэффициенты и полный числовой журнал в исключённой из Git папке `runs/`. В научном корпусе остаются обезличенные агрегаты и происхождение входов.